# Cross-Dataset Evaluation: CIC-IDS2017 Models → CIC-IDS2018 Data

This notebook evaluates the **CNN Classifier** and **CNN-Transformer** models
trained on CIC-IDS2017 against the **CIC-IDS2018** dataset.

This tests **generalization** — how well 2017-trained models detect attacks in
unseen 2018 traffic.  No retraining is performed; only inference + metrics.

**Workflow:**
1. Load 2017-trained checkpoints (`.pth`)
2. Discover & load CIC-IDS2018 CSV(s)
3. Apply the **same** preprocessing pipeline stored in the checkpoint
4. Run inference with F1-optimal threshold from validation
5. Display per-model metrics + side-by-side comparison
6. (Optional) Run SHAP on 2018 data

In [ ]:
# Cell 1: Setup & install
import os, sys, glob, warnings, importlib
from importlib.metadata import version, PackageNotFoundError

warnings.filterwarnings('ignore')
os.environ['TORCHDYNAMO_DISABLE'] = '1'

REPO_URL = 'https://github.com/samaraweeramethun-eng/CNN-Transformer.git'
REPO_DIR = '/kaggle/working/cnn_transformer_only'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Keep notebook repo in sync with latest pushed changes
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

!pip install -q shap lime joblib psutil scipy
!pip install -q -e .

SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Clear stale modules
for name in list(sys.modules):
    if name == 'cnn_transformer_only' or name.startswith('cnn_transformer_only.'):
        del sys.modules[name]

import torch
import numpy as np
import pandas as pd
cnn_transformer_only = importlib.import_module('cnn_transformer_only')

print('module file:', cnn_transformer_only.__file__)
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 1 — Locate 2017-trained checkpoints

In [ ]:
# Cell 2: Locate the 2017-trained checkpoints
# Update these paths if your checkpoints are saved elsewhere.
ARTIFACTS_DIR = '/kaggle/working/artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

CNN_CKPT_PATH = os.path.join(ARTIFACTS_DIR, 'cnn_classifier.pth')
CT_CKPT_PATH  = os.path.join(ARTIFACTS_DIR, 'cnn_transformer_ids.pth')

# Auto-search if not found at default location
for var_name, filename in [('CNN_CKPT_PATH', 'cnn_classifier.pth'),
                            ('CT_CKPT_PATH', 'cnn_transformer_ids.pth')]:
    path = eval(var_name)
    if not os.path.exists(path):
        hits = glob.glob(f'/kaggle/**/{filename}', recursive=True)
        if hits:
            path = hits[0]
            exec(f'{var_name} = path')
            print(f'Auto-found {filename}: {path}')
        else:
            print(f'WARNING: {filename} not found — upload it or update the path.')

print(f'CNN Classifier checkpoint : {CNN_CKPT_PATH}  (exists={os.path.exists(CNN_CKPT_PATH)})')
print(f'CNN-Transformer checkpoint: {CT_CKPT_PATH}  (exists={os.path.exists(CT_CKPT_PATH)})')

## Step 2 — Discover CIC-IDS2018 data

In [ ]:
# Cell 3: Discover CIC-IDS2018 CSV files under /kaggle/input
CICIDS2018_PATH = ''  # optional: set exact path to a CSV or folder

DATA_2018_CSVS = []

if CICIDS2018_PATH and os.path.exists(CICIDS2018_PATH):
    if os.path.isdir(CICIDS2018_PATH):
        DATA_2018_CSVS = sorted(glob.glob(os.path.join(CICIDS2018_PATH, '*.csv')))
    else:
        DATA_2018_CSVS = [CICIDS2018_PATH]
else:
    # Auto-detect 2018 CSVs
    patterns_2018 = [
        '/kaggle/input/**/*2018*.csv',
        '/kaggle/input/**/*CICFlowMeter*.csv',
        '/kaggle/input/**/*TrafficForML*.csv',
        '/kaggle/input/**/*.csv',
    ]
    for pattern in patterns_2018:
        candidates = sorted(glob.glob(pattern, recursive=True))
        # Exclude any 2017 files
        candidates = [c for c in candidates if '2017' not in c.lower()]
        if candidates:
            # Check if they are in the same directory
            dirs = set(os.path.dirname(c) for c in candidates)
            if len(dirs) == 1:
                DATA_2018_CSVS = candidates
            else:
                DATA_2018_CSVS = candidates
            break

print(f'Found {len(DATA_2018_CSVS)} CIC-IDS2018 CSV file(s):')
total_size_mb = 0
for f in DATA_2018_CSVS:
    sz = os.path.getsize(f) / 1024**2
    total_size_mb += sz
    print(f'  {os.path.basename(f):60s}  {sz:8.1f} MB')
print(f'  Total: {total_size_mb:.1f} MB')

if not DATA_2018_CSVS:
    raise FileNotFoundError(
        'No CIC-IDS2018 CSV files found. '
        'Please attach a Kaggle dataset or set CICIDS2018_PATH manually.'
    )

# Preview first file
peek = pd.read_csv(DATA_2018_CSVS[0], nrows=5)
peek.columns = [str(c).strip() for c in peek.columns]
label_candidates = [c for c in peek.columns if 'label' in c.lower()]
print(f'\nPreview columns: {len(peek.columns)}')
print(f'Label column: {label_candidates[0] if label_candidates else "NOT FOUND"}')
peek.head()

## Step 3 — Load checkpoints & reconstruct preprocessors

In [ ]:
# Cell 4: Load both checkpoints and rebuild models + preprocessors
import torch.nn.functional as F
from cnn_transformer_only.interpretability.shap_runner import (
    load_checkpoint,
    build_model_from_ckpt,
    resolve_preprocessor,
)
from cnn_transformer_only.data import (
    calculate_comprehensive_metrics,
    binary_predictions_from_proba,
    detect_label_column,
)
from cnn_transformer_only.utils.device import setup_device

device, _ = setup_device()
print(f'Device: {device}')

# ── Load CNN Classifier ──
cnn_ckpt = load_checkpoint(CNN_CKPT_PATH)
cnn_model = build_model_from_ckpt(cnn_ckpt, device)
cnn_feature_cols = cnn_ckpt['feature_columns']
cnn_preprocessor = resolve_preprocessor(cnn_ckpt.get('preprocessor'), cnn_feature_cols)
cnn_threshold = cnn_ckpt.get('best_threshold', 0.5)
cnn_csv_cols = cnn_ckpt.get('preprocessor', {}).get('csv_columns', cnn_feature_cols)
print(f'\nCNN Classifier loaded: {len(cnn_feature_cols)} features, threshold={cnn_threshold:.4f}')
print(f'  2017 test metrics: {cnn_ckpt.get("test_metrics", {})}')

# ── Load CNN-Transformer ──
ct_ckpt = load_checkpoint(CT_CKPT_PATH)
ct_model = build_model_from_ckpt(ct_ckpt, device)
ct_feature_cols = ct_ckpt['feature_columns']
ct_preprocessor = resolve_preprocessor(ct_ckpt.get('preprocessor'), ct_feature_cols)
ct_threshold = ct_ckpt.get('best_threshold', 0.5)
ct_csv_cols = ct_ckpt.get('preprocessor', {}).get('csv_columns', ct_feature_cols)
print(f'\nCNN-Transformer loaded: {len(ct_feature_cols)} features, threshold={ct_threshold:.4f}')
print(f'  2017 test metrics: {ct_ckpt.get("test_metrics", {})}')

## Step 4 — Load & preprocess CIC-IDS2018 data

In [ ]:
# Cell 5: Load CIC-IDS2018 data with chunked reading
import gc

CHUNKSIZE = 200_000
MAX_ROWS  = 0  # 0 = load all rows

# Determine label column from 2018 data
header_df = pd.read_csv(DATA_2018_CSVS[0], nrows=0)
header_df.columns = [str(c).strip() for c in header_df.columns]
label_col_2018 = detect_label_column(header_df)
print(f'2018 label column: {label_col_2018}')

# Use the CNN-Transformer's csv_columns as the reference feature set
# (both models use the same features when trained together)
reference_csv_cols = list(ct_csv_cols)

# Check which reference features exist in 2018 data
available_cols_2018 = set(header_df.columns)
matched = [c for c in reference_csv_cols if c in available_cols_2018]
missing = [c for c in reference_csv_cols if c not in available_cols_2018]

print(f'Reference features (from 2017 model): {len(reference_csv_cols)}')
print(f'Matched in 2018 data: {len(matched)}')
if missing:
    print(f'Missing in 2018 (will be zero-filled): {len(missing)}')
    for m in missing[:10]:
        print(f'  - {m}')
    if len(missing) > 10:
        print(f'  ... and {len(missing)-10} more')

# Extra columns in 2018 not in 2017 (informational)
blacklist = {label_col_2018, 'Flow ID', 'Source IP', 'Destination IP',
             'Src IP', 'Dst IP', 'Timestamp'}
extra_2018 = [c for c in available_cols_2018
              if c not in set(reference_csv_cols) and c not in blacklist]
if extra_2018:
    print(f'\nExtra columns in 2018 (ignored): {len(extra_2018)}')
    for e in extra_2018[:5]:
        print(f'  + {e}')
    if len(extra_2018) > 5:
        print(f'  ... and {len(extra_2018)-5} more')

In [ ]:
# Cell 6: Chunked load of 2018 data
usecols = [c for c in (matched + [label_col_2018]) if c in available_cols_2018]
usecols = list(dict.fromkeys(usecols))  # preserve order, deduplicate

x_parts = []
y_parts = []
label_dist = {}
rows_loaded = 0

use_cap = MAX_ROWS > 0

for csv_path in DATA_2018_CSVS:
    print(f'Reading: {os.path.basename(csv_path)}')
    for chunk in pd.read_csv(csv_path, usecols=usecols, low_memory=False,
                             chunksize=CHUNKSIZE):
        chunk.columns = [str(c).strip() for c in chunk.columns]

        # Binary labels
        labels_raw = chunk[label_col_2018].astype(str).str.strip().str.upper()
        y_chunk = (labels_raw != 'BENIGN').astype(np.int8).to_numpy()

        # Count per-label distribution
        for lbl in labels_raw:
            label_dist[lbl] = label_dist.get(lbl, 0) + 1

        # Feature extraction
        x_df = chunk.reindex(columns=matched)
        obj_cols = x_df.select_dtypes(include=['object']).columns
        if len(obj_cols) > 0:
            x_df = x_df.copy()
            for col in obj_cols:
                x_df[col] = pd.to_numeric(x_df[col], errors='coerce')

        # Replace inf with NaN (preprocessing will handle NaN)
        x_arr = x_df.to_numpy(dtype=np.float32)
        x_arr[np.isinf(x_arr)] = np.nan

        x_parts.append(x_arr)
        y_parts.append(y_chunk)
        rows_loaded += len(y_chunk)

        del chunk, x_df, x_arr, y_chunk

        if use_cap and rows_loaded >= MAX_ROWS:
            break
    if use_cap and rows_loaded >= MAX_ROWS:
        break

X_2018_raw = np.concatenate(x_parts, axis=0)
y_2018     = np.concatenate(y_parts, axis=0)
if use_cap and len(y_2018) > MAX_ROWS:
    X_2018_raw = X_2018_raw[:MAX_ROWS]
    y_2018     = y_2018[:MAX_ROWS]

del x_parts, y_parts
gc.collect()

benign_count = int((y_2018 == 0).sum())
attack_count = int((y_2018 == 1).sum())
print(f'\n=== CIC-IDS2018 Loaded ===')
print(f'Total rows: {len(y_2018):,}')
print(f'Features (matched): {X_2018_raw.shape[1]}')
print(f'BENIGN: {benign_count:,} ({benign_count/len(y_2018)*100:.1f}%)')
print(f'ATTACK: {attack_count:,} ({attack_count/len(y_2018)*100:.1f}%)')

print(f'\nAttack label breakdown (top 15):')
for lbl, cnt in sorted(label_dist.items(), key=lambda x: -x[1])[:15]:
    print(f'  {lbl:35s}: {cnt:>10,}')

In [ ]:

# Cell 7: Apply the 2017-trained preprocessor to 2018 data

def preprocess_for_model(X_raw, matched_cols, preprocessor, full_feature_cols):
    """Apply the saved 2017 preprocessing pipeline to raw 2018 features."""
    df = pd.DataFrame(X_raw, columns=matched_cols)
    # Fill NaN/inf before transform so the preprocessor doesn't propagate them
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    X_processed = preprocessor.transform(df)
    # Safety: replace any NaN/inf that the transform itself may introduce
    X_processed = np.nan_to_num(X_processed, nan=0.0, posinf=0.0, neginf=0.0)
    return X_processed.astype(np.float32)

print('Preprocessing 2018 data for CNN Classifier...')
X_2018_cnn = preprocess_for_model(X_2018_raw, matched, cnn_preprocessor, cnn_feature_cols)
print(f'  Shape: {X_2018_cnn.shape}')

print('Preprocessing 2018 data for CNN-Transformer...')
X_2018_ct = preprocess_for_model(X_2018_raw, matched, ct_preprocessor, ct_feature_cols)
print(f'  Shape: {X_2018_ct.shape}')

# Free raw data
del X_2018_raw
gc.collect()
print('Done.')


## Step 5 — Inference on CIC-IDS2018

In [ ]:

# Cell 8: Batch inference function (optimised for large datasets)
from torch.utils.data import DataLoader, TensorDataset

@torch.no_grad()
def run_inference(model, X, y, batch_size=2048, device=device, use_amp=True):
    """Run model inference and return probabilities + predictions.
    
    Args:
        use_amp: Use automatic mixed-precision (float16) on CUDA for ~2-4× speedup.
    """
    model.eval()
    # Sanitise inputs: replace NaN/inf with 0 so the model never sees them
    X_clean = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    dataset = TensorDataset(torch.FloatTensor(X_clean), torch.LongTensor(y))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=torch.cuda.is_available())

    all_probs = []
    all_targets = []
    total_batches = len(loader)
    use_amp = use_amp and device.type == 'cuda'

    for i, (data, target) in enumerate(loader):
        data = data.to(device, non_blocking=True)

        if use_amp:
            with torch.amp.autocast('cuda'):
                logits = model(data)
        else:
            logits = model(data)

        probs = F.softmax(logits.float(), dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_targets.append(target.numpy())

        if (i + 1) % max(1, total_batches // 10) == 0 or (i + 1) == total_batches:
            print(f'  batch {i+1}/{total_batches} '
                  f'({(i+1)/total_batches*100:.0f}%)', flush=True)

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    return y_prob, y_true

print('Inference function ready.')


In [ ]:
# Cell 9: Run inference — CNN Classifier on 2018
import time

print('Running CNN Classifier inference on CIC-IDS2018...')
t0 = time.time()
cnn_probs_2018, cnn_y_true = run_inference(cnn_model, X_2018_cnn, y_2018)
cnn_preds_2018 = binary_predictions_from_proba(cnn_probs_2018, threshold=cnn_threshold)
cnn_metrics_2018 = calculate_comprehensive_metrics(cnn_y_true, cnn_preds_2018, cnn_probs_2018)

print(f'Elapsed: {time.time()-t0:.1f}s')
print(f'Threshold (from 2017 val): {cnn_threshold:.4f}')
print(f'\n=== CNN Classifier — CIC-IDS2018 Metrics ===')
for k in ['accuracy', 'auc_roc', 'auc_pr', 'f1_score', 'precision', 'recall']:
    print(f'  {k:12s}: {cnn_metrics_2018[k]:.4f}')

In [ ]:

# Cell 10: Run inference — CNN-Transformer on 2018
# NOTE: CNN-Transformer is ~20-40× heavier per sample than the CNN Classifier
# because of multi-head self-attention across all feature tokens.
# Mixed-precision (AMP) + larger batch gives a major speedup on GPU.
print('Running CNN-Transformer inference on CIC-IDS2018...')

# Use half-precision model weights on GPU for additional speed
if device.type == 'cuda':
    ct_model.half()
    ct_model.float()  # keep parameters float32 — AMP autocast handles the rest

t0 = time.time()
ct_probs_2018, ct_y_true = run_inference(
    ct_model, X_2018_ct, y_2018,
    batch_size=4096,   # larger batches saturate GPU better
    use_amp=True,      # float16 on CUDA — ~2-4× faster
)
ct_preds_2018 = binary_predictions_from_proba(ct_probs_2018, threshold=ct_threshold)
ct_metrics_2018 = calculate_comprehensive_metrics(ct_y_true, ct_preds_2018, ct_probs_2018)

print(f'Elapsed: {time.time()-t0:.1f}s')
print(f'Threshold (from 2017 val): {ct_threshold:.4f}')
print(f'\n=== CNN-Transformer — CIC-IDS2018 Metrics ===')
for k in ['accuracy', 'auc_roc', 'auc_pr', 'f1_score', 'precision', 'recall']:
    print(f'  {k:12s}: {ct_metrics_2018[k]:.4f}')


## Step 6 — Side-by-Side Comparison (2017 vs 2018)

In [ ]:
# Cell 11: Full comparison table
import matplotlib.pyplot as plt

cnn_2017_metrics = cnn_ckpt.get('test_metrics', {})
ct_2017_metrics  = ct_ckpt.get('test_metrics', {})

metric_keys = ['accuracy', 'auc_roc', 'auc_pr', 'f1_score', 'precision', 'recall']

comparison = pd.DataFrame({
    'Metric': metric_keys,
    'CNN (2017 test)': [cnn_2017_metrics.get(k, None) for k in metric_keys],
    'CNN (2018 test)': [cnn_metrics_2018.get(k, None) for k in metric_keys],
    'CT (2017 test)':  [ct_2017_metrics.get(k, None) for k in metric_keys],
    'CT (2018 test)':  [ct_metrics_2018.get(k, None) for k in metric_keys],
}).set_index('Metric')

comparison['CNN Δ (2018−2017)'] = comparison['CNN (2018 test)'] - comparison['CNN (2017 test)']
comparison['CT Δ (2018−2017)']  = comparison['CT (2018 test)']  - comparison['CT (2017 test)']

print('=' * 90)
print('CROSS-DATASET GENERALIZATION: Trained on 2017 → Tested on 2018')
print('=' * 90)
display(comparison)

# ── Bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, model_name, cols in [
    (axes[0], 'CNN Classifier', ['CNN (2017 test)', 'CNN (2018 test)']),
    (axes[1], 'CNN-Transformer', ['CT (2017 test)', 'CT (2018 test)']),
]:
    comparison[cols].plot.bar(ax=ax, rot=25, color=['#2196F3', '#FF9800'])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(f'{model_name}: 2017 vs 2018')
    ax.legend(loc='lower right')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=7)

plt.suptitle('Generalization: Trained on CIC-IDS2017 → Evaluated on CIC-IDS2018',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'cross_dataset_2017_to_2018.png'),
            dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12: Model-vs-model comparison on 2018 data
model_cmp_2018 = pd.DataFrame({
    'Metric': metric_keys,
    'CNN Classifier': [cnn_metrics_2018[k] for k in metric_keys],
    'CNN-Transformer': [ct_metrics_2018[k] for k in metric_keys],
}).set_index('Metric')

model_cmp_2018['Δ (CT − CNN)'] = model_cmp_2018['CNN-Transformer'] - model_cmp_2018['CNN Classifier']

print('=' * 65)
print('MODEL COMPARISON on CIC-IDS2018 (unseen dataset)')
print('=' * 65)
display(model_cmp_2018)

print('\nPer-metric winner (on 2018):')
for metric in metric_keys:
    cnn_val = cnn_metrics_2018[metric]
    ct_val  = ct_metrics_2018[metric]
    winner = 'CNN-Transformer' if ct_val > cnn_val else (
        'CNN Classifier' if cnn_val > ct_val else 'Tie')
    print(f'  {metric:12s}: {winner} ({ct_val:.4f} vs {cnn_val:.4f})')

## Step 7 — Confusion Matrices

In [ ]:
# Cell 13: Confusion matrices for both models on 2018
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name, y_pred in [
    (axes[0], 'CNN Classifier', cnn_preds_2018),
    (axes[1], 'CNN-Transformer', ct_preds_2018),
]:
    cm = confusion_matrix(y_2018, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['BENIGN', 'ATTACK'])
    disp.plot(ax=ax, cmap='Blues', values_format=',')
    ax.set_title(f'{name} on CIC-IDS2018')

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'confusion_2018.png'),
            dpi=160, bbox_inches='tight')
plt.show()

# Print raw numbers
for name, y_pred in [('CNN Classifier', cnn_preds_2018),
                      ('CNN-Transformer', ct_preds_2018)]:
    cm = confusion_matrix(y_2018, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f'\n{name}:')
    print(f'  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
    print(f'  FPR={fp/(fp+tn)*100:.2f}%  FNR={fn/(fn+tp)*100:.2f}%')

## Step 8 — ROC & Precision-Recall Curves

In [ ]:
# Cell 14: ROC and PR curves for both models on 2018
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sanitise probabilities (replace any residual NaN/inf with 0)
cnn_probs_clean = np.nan_to_num(cnn_probs_2018, nan=0.0, posinf=1.0, neginf=0.0)
ct_probs_clean  = np.nan_to_num(ct_probs_2018,  nan=0.0, posinf=1.0, neginf=0.0)

# ROC
ax = axes[0]
for name, y_prob, color in [
    ('CNN Classifier', cnn_probs_clean, '#2196F3'),
    ('CNN-Transformer', ct_probs_clean, '#FF9800'),
]:
    fpr, tpr, _ = roc_curve(y_2018, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.4f})', color=color, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — CIC-IDS2018')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# PR
ax = axes[1]
for name, y_prob, color in [
    ('CNN Classifier', cnn_probs_clean, '#2196F3'),
    ('CNN-Transformer', ct_probs_clean, '#FF9800'),
]:
    prec, rec, _ = precision_recall_curve(y_2018, y_prob)
    pr_auc = auc(rec, prec)
    ax.plot(rec, prec, label=f'{name} (AUC={pr_auc:.4f})', color=color, lw=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — CIC-IDS2018')
ax.legend(loc='lower left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'roc_pr_2018.png'),
            dpi=160, bbox_inches='tight')
plt.show()

## Step 9 — SHAP & LIME Explainability on CIC-IDS2018

In [ ]:
# Cell 15: SHAP analysis on 2018 data (uses in-memory models & preprocessed data)
import shap

SHAP_BG   = 1000   # background samples for GradientExplainer
SHAP_EVAL = 1000   # samples to explain
shap_seed = 42
rng_shap  = np.random.RandomState(shap_seed)

# Wrapper to ensure model returns plain logits
class LogitsOnly(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x):
        out = self.base(x)
        return out[0] if isinstance(out, (tuple, list)) else out

shap_results = {}

for label, model, X_data, feature_cols, ckpt_path in [
    ('CNN Classifier',  cnn_model, X_2018_cnn, cnn_feature_cols, CNN_CKPT_PATH),
    ('CNN-Transformer', ct_model,  X_2018_ct,  ct_feature_cols,  CT_CKPT_PATH),
]:
    print(f'\n{"="*60}')
    print(f'SHAP — {label} on CIC-IDS2018')
    print(f'{"="*60}')

    # Sample indices
    n = X_data.shape[0]
    pool_idx = rng_shap.choice(n, size=min(max(SHAP_BG, SHAP_EVAL), n), replace=False)
    X_pool = np.nan_to_num(X_data[pool_idx], nan=0.0, posinf=0.0, neginf=0.0)

    bg_tensor   = torch.FloatTensor(X_pool[:SHAP_BG]).to(device)
    eval_tensor = torch.FloatTensor(X_pool[:SHAP_EVAL]).to(device)

    logits_model = LogitsOnly(model).to(device)
    logits_model.eval()
    explainer = shap.GradientExplainer(logits_model, bg_tensor)

    # Compute SHAP in chunks
    sv_chunks = []
    for start in range(0, eval_tensor.shape[0], 256):
        end = min(start + 256, eval_tensor.shape[0])
        raw_sv = explainer.shap_values(eval_tensor[start:end])
        if isinstance(raw_sv, (list, tuple)):
            sv_chunks.append(raw_sv[1])
        elif isinstance(raw_sv, np.ndarray) and raw_sv.ndim == 3:
            sv_chunks.append(raw_sv[:, :, 1])
        else:
            sv_chunks.append(raw_sv)

    sv = np.concatenate(sv_chunks, axis=0)
    mean_abs = np.abs(sv).mean(axis=0)

    # Save CSV
    out_dir = os.path.join(ARTIFACTS_DIR, 'shap_2018', label.lower().replace(' ', '_').replace('-', '_'))
    os.makedirs(out_dir, exist_ok=True)
    imp_df = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': mean_abs})
    imp_df = imp_df.sort_values('mean_abs_shap', ascending=False)
    csv_path = os.path.join(out_dir, 'shap_global_importance_attack.csv')
    imp_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    display(imp_df.head(15))

    # Summary plot
    shap.summary_plot(
        sv, features=eval_tensor.detach().cpu().numpy(),
        feature_names=feature_cols, show=False,
    )
    plt.tight_layout()
    summary_path = os.path.join(out_dir, 'shap_summary_attack.png')
    plt.savefig(summary_path, dpi=160, bbox_inches='tight')
    plt.show()

    # Waterfall for highest-confidence attack sample
    with torch.no_grad():
        base_val = float(logits_model(bg_tensor)[:, 1].mean().cpu().item())
    probs_eval = torch.softmax(logits_model(eval_tensor), dim=1)[:, 1].detach().cpu().numpy()
    target_idx = int(np.argmax(probs_eval))
    explanation = shap.Explanation(
        values=sv[target_idx], base_values=base_val,
        data=eval_tensor[target_idx].detach().cpu().numpy(),
        feature_names=feature_cols,
    )
    shap.plots.waterfall(explanation, max_display=20, show=False)
    plt.tight_layout()
    waterfall_path = os.path.join(out_dir, 'shap_waterfall_attack.png')
    plt.savefig(waterfall_path, dpi=160, bbox_inches='tight')
    plt.show()

    shap_results[label] = imp_df
    print(f'Done — {label}')

print('\nSHAP analysis complete for both models.')

In [ ]:
# Cell 16: LIME explanations on 2018 data
!pip install -q lime

import lime
import lime.lime_tabular

LIME_SAMPLES   = 500   # number of samples to explain for global importance
LIME_NUM_FEATS = 20    # top features to display per explanation
lime_seed      = 42
rng_lime       = np.random.RandomState(lime_seed)

lime_results = {}

for label, model, X_data, feature_cols in [
    ('CNN Classifier',  cnn_model, X_2018_cnn, cnn_feature_cols),
    ('CNN-Transformer', ct_model,  X_2018_ct,  ct_feature_cols),
]:
    print(f'\n{"="*60}')
    print(f'LIME — {label} on CIC-IDS2018')
    print(f'{"="*60}')

    # Sanitise
    X_clean = np.nan_to_num(X_data, nan=0.0, posinf=0.0, neginf=0.0)

    # Build predict_proba function for LIME
    _model = model
    _device = device

    def predict_proba_fn(X_np, _m=_model, _d=_device):
        _m.eval()
        X_t = torch.FloatTensor(X_np).to(_d)
        with torch.no_grad():
            logits = _m(X_t)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            probs = F.softmax(logits.float(), dim=1)
        return probs.cpu().numpy()

    # Create LIME explainer
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_clean[rng_lime.choice(len(X_clean), size=min(5000, len(X_clean)), replace=False)],
        feature_names=feature_cols,
        class_names=['BENIGN', 'ATTACK'],
        mode='classification',
        random_state=lime_seed,
    )

    # --- Global feature importance via aggregated LIME ---
    sample_idx = rng_lime.choice(len(X_clean), size=min(LIME_SAMPLES, len(X_clean)), replace=False)
    feat_importance = np.zeros(len(feature_cols))

    print(f'Explaining {len(sample_idx)} samples...')
    for count, idx in enumerate(sample_idx):
        exp = explainer.explain_instance(
            X_clean[idx], predict_proba_fn,
            num_features=len(feature_cols), labels=(1,),
        )
        for feat_idx, weight in exp.as_map()[1]:
            feat_importance[feat_idx] += abs(weight)
        if (count + 1) % 100 == 0:
            print(f'  {count+1}/{len(sample_idx)}', flush=True)

    feat_importance /= len(sample_idx)

    # Save CSV
    out_dir = os.path.join(ARTIFACTS_DIR, 'lime_2018', label.lower().replace(' ', '_').replace('-', '_'))
    os.makedirs(out_dir, exist_ok=True)
    lime_df = pd.DataFrame({'feature': feature_cols, 'mean_abs_lime_weight': feat_importance})
    lime_df = lime_df.sort_values('mean_abs_lime_weight', ascending=False)
    csv_path = os.path.join(out_dir, 'lime_global_importance.csv')
    lime_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    display(lime_df.head(15))

    # Bar chart of top features
    top_n = lime_df.head(LIME_NUM_FEATS)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_n['feature'][::-1], top_n['mean_abs_lime_weight'][::-1], color='#4CAF50')
    ax.set_xlabel('Mean |LIME weight|')
    ax.set_title(f'LIME Global Feature Importance — {label} (2018)')
    plt.tight_layout()
    bar_path = os.path.join(out_dir, 'lime_global_importance.png')
    plt.savefig(bar_path, dpi=160, bbox_inches='tight')
    plt.show()

    lime_results[label] = lime_df
    print(f'Done — {label}')

print('\nLIME analysis complete for both models.')

In [ ]:
# Cell 17: LIME — single-instance explanations (highest-confidence attack sample)

for label, model, X_data, feature_cols, probs_arr in [
    ('CNN Classifier',  cnn_model, X_2018_cnn, cnn_feature_cols, cnn_probs_2018),
    ('CNN-Transformer', ct_model,  X_2018_ct,  ct_feature_cols,  ct_probs_2018),
]:
    print(f'\n{"="*60}')
    print(f'LIME Single-Instance — {label}')
    print(f'{"="*60}')

    X_clean = np.nan_to_num(X_data, nan=0.0, posinf=0.0, neginf=0.0)

    _model = model
    _device = device

    def predict_proba_fn(X_np, _m=_model, _d=_device):
        _m.eval()
        X_t = torch.FloatTensor(X_np).to(_d)
        with torch.no_grad():
            logits = _m(X_t)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            probs = F.softmax(logits.float(), dim=1)
        return probs.cpu().numpy()

    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_clean[rng_lime.choice(len(X_clean), size=min(5000, len(X_clean)), replace=False)],
        feature_names=feature_cols,
        class_names=['BENIGN', 'ATTACK'],
        mode='classification',
        random_state=lime_seed,
    )

    # Pick highest-confidence attack sample
    probs_clean = np.nan_to_num(probs_arr, nan=0.0)
    attack_mask = (y_2018 == 1)
    if attack_mask.any():
        attack_probs = probs_clean.copy()
        attack_probs[~attack_mask] = -1
        target_idx = int(np.argmax(attack_probs))
    else:
        target_idx = int(np.argmax(probs_clean))

    print(f'Explaining sample index {target_idx} (prob={probs_clean[target_idx]:.4f}, '
          f'true_label={"ATTACK" if y_2018[target_idx]==1 else "BENIGN"})')

    exp = explainer.explain_instance(
        X_clean[target_idx], predict_proba_fn,
        num_features=LIME_NUM_FEATS, labels=(1,),
    )

    # Show as matplotlib figure
    fig = exp.as_pyplot_figure(label=1)
    fig.set_size_inches(12, 6)
    fig.suptitle(f'LIME Explanation — {label} (Attack Sample)', fontsize=13)
    plt.tight_layout()
    out_dir = os.path.join(ARTIFACTS_DIR, 'lime_2018', label.lower().replace(' ', '_').replace('-', '_'))
    os.makedirs(out_dir, exist_ok=True)
    fig_path = os.path.join(out_dir, 'lime_single_attack.png')
    plt.savefig(fig_path, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

print('\nLIME single-instance explanations complete.')

In [ ]:
# Cell 18: SHAP vs LIME — side-by-side top-feature comparison
TOP_K = 15

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for col, label in enumerate(['CNN Classifier', 'CNN-Transformer']):
    shap_df = shap_results[label].head(TOP_K).copy()
    lime_df = lime_results[label].head(TOP_K).copy()

    # SHAP bar
    ax = axes[0][col]
    ax.barh(shap_df['feature'][::-1],
            shap_df['mean_abs_shap'][::-1], color='#2196F3')
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'SHAP Top-{TOP_K} — {label}')

    # LIME bar
    ax = axes[1][col]
    ax.barh(lime_df['feature'][::-1],
            lime_df['mean_abs_lime_weight'][::-1], color='#4CAF50')
    ax.set_xlabel('Mean |LIME weight|')
    ax.set_title(f'LIME Top-{TOP_K} — {label}')

plt.suptitle('SHAP vs LIME Feature Importance — CIC-IDS2018', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'shap_vs_lime_2018.png'),
            dpi=160, bbox_inches='tight')
plt.show()

# Print overlap between SHAP and LIME top features
for label in ['CNN Classifier', 'CNN-Transformer']:
    shap_top = set(shap_results[label].head(TOP_K)['feature'])
    lime_top = set(lime_results[label].head(TOP_K)['feature'])
    overlap = shap_top & lime_top
    print(f'\n{label} — Top-{TOP_K} overlap: {len(overlap)}/{TOP_K} features agree')
    if overlap:
        print(f'  Common: {sorted(overlap)}')

## Summary

In [ ]:
# Cell 16: Final summary
print('=' * 70)
print('CROSS-DATASET EVALUATION SUMMARY')
print('  Training data:  CIC-IDS2017')
print(f'  Testing data:   CIC-IDS2018 ({len(y_2018):,} rows)')
print(f'  Attack ratio:   {attack_count/len(y_2018)*100:.1f}%')
print('=' * 70)

summary = pd.DataFrame({
    'Metric': metric_keys,
    'CNN→2017': [cnn_2017_metrics.get(k, 0) for k in metric_keys],
    'CNN→2018': [cnn_metrics_2018[k] for k in metric_keys],
    'CT→2017':  [ct_2017_metrics.get(k, 0) for k in metric_keys],
    'CT→2018':  [ct_metrics_2018[k] for k in metric_keys],
}).set_index('Metric')

display(summary.style.format('{:.4f}').highlight_max(axis=1, color='lightgreen'))

print('\nInterpretation:')
for name, m17, m18 in [('CNN Classifier', cnn_2017_metrics, cnn_metrics_2018),
                        ('CNN-Transformer', ct_2017_metrics, ct_metrics_2018)]:
    f1_17 = m17.get('f1_score', 0)
    f1_18 = m18['f1_score']
    drop = f1_17 - f1_18
    if drop > 0.05:
        verdict = f'F1 dropped by {drop:.4f} — possible generalization gap'
    elif drop > 0:
        verdict = f'F1 dropped slightly by {drop:.4f} — reasonable generalization'
    else:
        verdict = f'F1 improved by {-drop:.4f} — strong generalization'
    print(f'  {name}: {verdict}')

print('\nSaved artifacts:')
for f in ['cross_dataset_2017_to_2018.png', 'confusion_2018.png', 'roc_pr_2018.png']:
    path = os.path.join(ARTIFACTS_DIR, f)
    print(f'  {"✓" if os.path.exists(path) else "✗"} {path}')